<a href="https://colab.research.google.com/github/r2rgen/R2RGen/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install \
    numpy \
    scipy \
    opencv-python \
    matplotlib==3.9 \
    open3d \
    trimesh \
    imageio \
    pathos \
    cupy-cuda12x \
    ninja

# Install nvdiffrast separately from its GitHub repository
!pip install git+https://github.com/NVlabs/nvdiffrast

  Cloning https://github.com/NVlabs/nvdiffrast to /tmp/pip-req-build-pjzo797t
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast /tmp/pip-req-build-pjzo797t
  Resolved https://github.com/NVlabs/nvdiffrast to commit 729261dc64c4241ea36efda84fbf532cc8b425b8
  Preparing metadata (setup.py) ... done


In [2]:
# =============================================================================
# 2. FILE DOWNLOAD AND EXTRACTION
# =============================================================================
import os
import tarfile
import zipfile
import gdown

def download_and_extract(file_id, output_path, extract_to='.'):
    print(f"Downloading: {os.path.basename(output_path)}")

    os.makedirs(extract_to, exist_ok=True)

    try:
        gdown.download(id=file_id, output=output_path, quiet=False)
        print(f"Downloaded: {output_path}")

        print(f"Extracting to '{extract_to}'...")
        if output_path.endswith('.zip'):
            with zipfile.ZipFile(output_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
        elif output_path.endswith('.tar.gz') or output_path.endswith('.tgz'):
            with tarfile.open(output_path, 'r:gz') as tar_ref:
                tar_ref.extractall(extract_to)
        print(f"Extracted successfully.")

        return True

    except Exception as e:
        print(f"Download or extraction failed: {e}")
        return False

    finally:
        if os.path.exists(output_path):
            os.remove(output_path)
            print(f"Cleaned up: {output_path}")

DEMO_DATA_FILE_ID = "14Sw3-JSvgUBdiRQUGkFmndGvNvc0OkFb"
GRIPPER_CLOUDS_FILE_ID = "1gaYNbm16c2N12EqgqKfHPiHSIWfcdrCu"

print("\nDownloading and extracting demo data...")
success1 = download_and_extract(DEMO_DATA_FILE_ID, "demo_data_complete.tar.gz")

print("\nDownloading and extracting gripper clouds...")
success2 = download_and_extract(GRIPPER_CLOUDS_FILE_ID, "gripper_clouds.zip", extract_to='debug/gripper_clouds')

if success1 and success2:
    print("\nAll data files are ready!")
else:
    print("\nSome downloads failed.")


Downloading: demo_data_complete.tar.gz


Downloading...
From (original): https://drive.google.com/uc?id=14Sw3-JSvgUBdiRQUGkFmndGvNvc0OkFb
From (redirected): https://drive.google.com/uc?id=14Sw3-JSvgUBdiRQUGkFmndGvNvc0OkFb&confirm=t&uuid=bf2ba42c-15b3-4b8e-a6f8-5a7d7541cae2
To: /content/demo_data_complete.tar.gz
100%|██████████| 164M/164M [00:01<00:00, 132MB/s]
/tmp/ipython-input-4288338312.py:24: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar_ref.extractall(extract_to)


Downloaded: demo_data_complete.tar.gz
Extracting to '.'...
Extracted successfully.
Cleaned up: demo_data_complete.tar.gz

Downloading: gripper_clouds.zip


Downloading...
From (original): https://drive.google.com/uc?id=1gaYNbm16c2N12EqgqKfHPiHSIWfcdrCu
From (redirected): https://drive.google.com/uc?id=1gaYNbm16c2N12EqgqKfHPiHSIWfcdrCu&confirm=t&uuid=65c06c36-b6c6-4f4f-a085-1f053d2b875d
To: /content/gripper_clouds.zip
100%|██████████| 62.5M/62.5M [00:00<00:00, 105MB/s]


Downloaded: gripper_clouds.zip
Extracting to 'debug/gripper_clouds'...
Extracted successfully.
Cleaned up: gripper_clouds.zip

All data files are ready!


## 1. Core Functions

In [3]:

from __future__ import annotations
import argparse, re, random, os
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import open3d as o3d
import trimesh
import cv2
import nvdiffrast.torch as dr
import glob
import time

def build_camera_to_robot(cam_xyz: np.ndarray, pitch_deg: float) -> np.ndarray:
    t_cam2base = np.array([-0.102, 0.125, 0.392])
    R_cam2base = np.array([[ 0.72798859, -0.31574097,  0.60855588],
                           [-0.68440788, -0.38677624,  0.61805339],
                           [ 0.04023018, -0.86643625, -0.49766431]])

    T_cam2base = compose(R_cam2base, t_cam2base)

    T_arm2robot = np.array([[1, 0, 0, 0.30],
                            [0, 1, 0, -0.335],
                            [0, 0, 1, 1.11],
                            [0, 0, 0, 1.0]], dtype=float)
    T_cam2robot = T_arm2robot @ T_cam2base

    return T_cam2robot

def load_eef_pose_robot_from_npy(npy_path: Path) -> np.ndarray:
    """.npyEEF pose，4×4"""
    data = np.load(npy_path)

    if len(data) >= 6:
        x, y, z = data[0], data[1], data[2]
        rx, ry, rz = data[3], data[4], data[5]

        t_arm = np.array([x, y, z], dtype=float)

        from scipy.spatial.transform import Rotation as R_spt
        R_arm = R_spt.from_euler('xyz', [rx, ry, rz], degrees=False).as_matrix()

        T_ARM_TO_ROBOT = np.array([[1, 0, 0, 0.30],
                                   [0, 1, 0, -0.335],
                                   [0, 0, 1, 1.11],
                                   [0, 0, 0, 1.0]], dtype=float)

        T_EE_arm = compose(R_arm, t_arm)

        return T_ARM_TO_ROBOT @ T_EE_arm
    else:
        raise ValueError(f"NPY {npy_path} ，6")


import math
from scipy.optimize import minimize
from typing import Optional


def parse_kv_map(s: str) -> Dict[str, str]:
    """alias=path,alias=path → dict"""
    mp = {}
    for seg in s.split(','):
        if not seg:
            continue
        key, val = seg.split('=')
        mp[key.strip()] = val.strip()
    return mp


_smooth = lambda a: 3*a*a - 2*a*a*a


def build_transform_cache(rules: List[dict], aliases: List[str],
                          frame_min:int, frame_max:int,
                          seed:int=42):
    """Return {alias: {frame: (t_vec, deg)}}"""
    random.seed(seed)
    cache: Dict[str, Dict[int, Tuple[np.ndarray,float]]] = {al:{} for al in aliases}
    Itrans = np.zeros(3); Ideg = 0.0


    init_tf: Dict[str, Tuple[np.ndarray, float]] = {al: (Itrans.copy(), Ideg) for al in aliases}
    first_ts: Dict[str, int] = {al: float('inf') for al in aliases}

    for r in rules:
        s, mode, param, als = r['start'], r['mode'], r['param'], r['aliases']
        if mode == 'T':
            dv, dg = np.array(param), 0.0
        else:
            dv, dg = np.zeros(3), param[0]
        for al in als:
            if s < first_ts[al]:
                first_ts[al] = s
                init_tf[al] = (dv.copy(), dg)
            elif s == first_ts[al]:
                curr_trans, curr_deg = init_tf[al]
                if mode == 'T':
                    new_trans = curr_trans + dv
                    init_tf[al] = (new_trans, curr_deg)
                else:
                    new_deg = curr_deg + dg
                    init_tf[al] = (curr_trans, new_deg)

    kfs: Dict[str, List[Tuple[int, np.ndarray, float]]] = {}
    for al in aliases:
        if al == 'gripper':
            kfs[al] = [(frame_min, Itrans, Ideg)]
        else:
            kfs[al] = [(frame_min, init_tf[al][0], init_tf[al][1])]
    for r in rules:
        s,e,als,mode,param = r['start'],r['end'],r['aliases'],r['mode'],r['param']
        if mode=='T':
            dv=np.array(param); dg=0.0
        else:
            dv=np.zeros(3); dg=param[0]
        for al in als:
            kfs.setdefault(al, []).extend([(s,dv,dg),(e,dv,dg)])

    final_tf: Dict[str, Tuple[np.ndarray, float]] = {al: (Itrans.copy(), Ideg) for al in aliases}
    last_ts: Dict[str, int] = {al: -1 for al in aliases}

    for r in rules:
        e, mode, param, als = r['end'], r['mode'], r['param'], r['aliases']
        if mode == 'T':
            dv, dg = np.array(param), 0.0
        else:
            dv, dg = np.zeros(3), param[0]
        for al in als:
            if e > last_ts[al]:
                last_ts[al] = e
                final_tf[al] = (dv.copy(), dg)
            elif e == last_ts[al]:
                curr_trans, curr_deg = final_tf[al]
                if mode == 'T':
                    new_trans = curr_trans + dv
                    final_tf[al] = (new_trans, curr_deg)
                else:
                    new_deg = curr_deg + dg
                    final_tf[al] = (curr_trans, new_deg)

    for al in aliases:
        if al == 'gripper':
            kfs[al].append((frame_max, Itrans, Ideg))
        else:
            kfs[al].append((frame_max, final_tf[al][0], final_tf[al][1]))

    for al in aliases:
        frame_transforms: Dict[int, Tuple[np.ndarray, float]] = {}
        for f, tv, dg in kfs[al]:
            if f in frame_transforms:
                existing_trans, existing_deg = frame_transforms[f]
                accumulated_trans = existing_trans + tv
                accumulated_deg = existing_deg + dg
                frame_transforms[f] = (accumulated_trans, accumulated_deg)
            else:
                frame_transforms[f] = (tv.copy(), dg)

        kfs_sorted = [(f, *frame_transforms[f]) for f in sorted(frame_transforms.keys())]

        for i in range(len(kfs_sorted) - 1):
            f0, t0, d0 = kfs_sorted[i]
            f1, t1, d1 = kfs_sorted[i + 1]

            const_span = any((r['start'] == f0 and r['end'] == f1 and al in r['aliases']) for r in rules)

            for f in range(f0, f1 + 1):
                if const_span:
                    cache[al][f] = (t0, d0)
                else:
                    if f == f0:
                        cache[al][f] = (t0, d0)
                    elif f == f1:
                        cache[al][f] = (t1, d1)
                    else:
                        a = (f - f0) / (f1 - f0)
                        s = _smooth(a)
                        cache[al][f] = ((1 - s) * t0 + s * t1, (1 - s) * d0 + s * d1)
    return cache


def axis_angle_to_mat_z(deg: float):
    th = np.deg2rad(deg)
    c,s = np.cos(th), np.sin(th)
    R = np.array([[c,-s,0],[s,c,0],[0,0,1]],dtype=float)
    return R

def axis_angle_to_mat_x(deg: float):
    """XRotationRotation"""
    th = np.deg2rad(deg)
    c,s = np.cos(th), np.sin(th)
    R = np.array([[1,0,0],[0,c,-s],[0,s,c]],dtype=float)
    return R

def compose(R: np.ndarray, t: np.ndarray):
    T=np.eye(4); T[:3,:3]=R; T[:3,3]=t; return T

def rotate_around_point(deg: float, center_point: np.ndarray):
    """ZRotation4x4

    Args:
        deg: Rotationdeg（deg）
        center_point: Rotation [x, y, z]

    Returns:
        4x4：Translation，Rotation，Translation
    """
    R = axis_angle_to_mat_z(deg)

    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = center_point - R @ center_point

    return T

def calculate_all_objects_center_at_frame(mesh_objs: Dict[str, any], pose_dirs: Dict[str, str],
                                         frame_id: int, T_CAM_ROBOT: np.ndarray) -> np.ndarray:
    """frames

    Args:
        mesh_objs: mesh
        pose_dirs: pose
        frame_id: frames
        T_CAM_ROBOT:

    Returns:
        （）
    """
    centers = []

    for object_alias in pose_dirs:
        center = get_object_center_at_frame(object_alias, mesh_objs, pose_dirs, frame_id, T_CAM_ROBOT)
        centers.append(center)

    if len(centers) == 0:
        return np.array([0.0, 0.0, 0.0])

    total_center = np.mean(centers, axis=0)

    return total_center

def get_object_center_at_frame(object_alias: str, mesh_objs: Dict[str, any],
                              pose_dirs: Dict[str, str], frame_id: int,
                              T_CAM_ROBOT: np.ndarray) -> np.ndarray:
    """frames（）

    Args:
        object_alias:
        mesh_objs: mesh
        pose_dirs: pose
        frame_id: frames
        T_CAM_ROBOT:

    Returns:

    """
    if object_alias in mesh_objs:
        mesh = mesh_objs[object_alias]
        mesh_center = np.array(mesh.centroid)
    else:
        mesh_center = np.array([0.0, 0.0, 0.0])

    frame_str = f"{frame_id:06d}"
    pose_path = Path(pose_dirs[object_alias]) / f"{frame_str}.txt"

    if pose_path.exists():
        T_obj_cam = np.loadtxt(pose_path)
        mesh_center_homo = np.append(mesh_center, 1.0)
        center_in_cam = (T_obj_cam @ mesh_center_homo)[:3]

        center_in_robot_homo = np.append(center_in_cam, 1.0)
        center_in_robot = (T_CAM_ROBOT @ center_in_robot_homo)[:3]

        return center_in_robot
    else:
        return np.array([0.0, 0.0, 0.0])

def analyze_rotation_pattern(rules: List[dict], aliases: List[str]) -> Dict[str, List[int]]:
    """Rotation，Rotationframes

    Args:
        rules:
        aliases:

    Returns:
        {: [Rotationframes]}
    """
    rotation_pattern = {}

    for alias in aliases:
        if alias == 'gripper':
            continue
        rotation_pattern[alias] = []

        for rule in rules:
            if rule['mode'] == 'R' and alias in rule['aliases']:
                rotation_pattern[alias].append((rule['start'], rule['end']))

        rotation_pattern[alias].sort(key=lambda x: x[0])

    return rotation_pattern

def get_rotation_center_with_interpolation(alias: str, frame_id: int, rotation_patterns: Dict[str, List[int]],
                                          first_rotation_completed: Dict[str, bool], frame_min: int,
                                          object_center: np.ndarray, fixed_center: np.ndarray) -> Tuple[np.ndarray, str]:
    """framesRotation，

    ：
    1. Rotation：
    2. RotationRotation：Rotation
    3. Rotation：

    Args:
        alias:
        frame_id: frames
        rotation_patterns: Rotation
        first_rotation_completed: Rotation
        frame_min: frames
        object_center:
        fixed_center: Rotation（）

    Returns:
        (Rotation, )
    """
    if alias not in rotation_patterns or len(rotation_patterns[alias]) == 0:
        return fixed_center, "Rotation"

    if first_rotation_completed.get(alias, False):
        return fixed_center, "Rotation-"

    first_rotation_start = rotation_patterns[alias][0][0]
    first_rotation_end = rotation_patterns[alias][0][1]

    if frame_min <= frame_id <= first_rotation_end:
        return object_center, "Rotation-"

    if len(rotation_patterns[alias]) > 1:
        second_rotation_start = rotation_patterns[alias][1][0]

        if first_rotation_end < frame_id < second_rotation_start:
            alpha = (frame_id - first_rotation_end) / (second_rotation_start - first_rotation_end)
            smooth_alpha = 3 * alpha * alpha - 2 * alpha * alpha * alpha

            interpolated_center = (1 - smooth_alpha) * object_center + smooth_alpha * fixed_center
            return interpolated_center, f"(α={smooth_alpha:.2f})"

        if frame_id >= second_rotation_start:
            return fixed_center, "Rotation-"
    else:
        return fixed_center, "Rotation-"

    return fixed_center, "-"

def get_gripper_companion_objects(rules: List[dict], frame_id: int, frame_min: int) -> Tuple[str, str, float]:
    """frames（）

    ：
    1. ，
    2. ，

    Args:
        rules:
        frame_id: frames
        frame_min: frames

    Returns:
        (, , )
        - =0:
        - =1:
        - 0<<1:
    """
    for rule in rules:
        if rule['start'] <= frame_id <= rule['end']:
            if 'gripper' in rule['aliases']:
                for alias in rule['aliases']:
                    if alias != 'gripper' and alias.startswith('obj'):
                        return alias, None, 0.0

    gripper_rules = []
    for rule in rules:
        if 'gripper' in rule['aliases']:
            for alias in rule['aliases']:
                if alias != 'gripper' and alias.startswith('obj'):
                    gripper_rules.append((rule['start'], rule['end'], alias))
                    break

    gripper_rules.sort(key=lambda x: x[0])

    for i in range(len(gripper_rules)):
        start_i, end_i, obj_i = gripper_rules[i]

        if frame_id < start_i:
            if i > 0:
                prev_start, prev_end, prev_obj = gripper_rules[i-1]

                if prev_obj != obj_i:
                    alpha = (frame_id - prev_end) / (start_i - prev_end)
                    smooth_alpha = 3 * alpha * alpha - 2 * alpha * alpha * alpha
                    return prev_obj, obj_i, smooth_alpha
                else:
                    return prev_obj, None, 0.0
            else:
                return obj_i, None, 0.0
        elif start_i <= frame_id <= end_i:
            return obj_i, None, 0.0
        elif i == len(gripper_rules) - 1:
            return obj_i, None, 0.0

    return None, None, 0.0



## 2. Configuration

You can configure USER_CONFIG_1 through USER_CONFIG_4 here. Configurations 1-3 represent the initial positions of the three objects, and configuration 4 represents the final position of the 'bridge'.

In [4]:
USER_CONFIG_1 = {'enabled': True, 'translation': [0.1, 0.05, 0], 'rotation': 50}
USER_CONFIG_2 = {'enabled': True, 'translation': [-0.02, 0, 0], 'rotation': 140}
USER_CONFIG_3 = {'enabled': True, 'translation': [-0.02, 0, 0], 'rotation': -10.0}
USER_CONFIG_4 = {'enabled': True, 'translation': [0.1, 0.05, 0], 'rotation': -60}

print("=" * 70)
print("Configuration")
print("=" * 70)
for i, cfg in enumerate([USER_CONFIG_1, USER_CONFIG_2, USER_CONFIG_3, USER_CONFIG_4], 1):
    status = "ON" if cfg['enabled'] else "OFF"
    print(f"\nConfig {i}: {status}")
    if cfg['enabled']:
        print(f"  T: {cfg['translation']} m")
        print(f"  R: {cfg['rotation']} deg")


Configuration

Config 1: ON
  T: [0.1, 0.05, 0] m
  R: 50 deg

Config 2: ON
  T: [-0.02, 0, 0] m
  R: 140 deg

Config 3: ON
  T: [-0.02, 0, 0] m
  R: -10.0 deg

Config 4: ON
  T: [0.1, 0.05, 0] m
  R: -60 deg


## 3. Generate Rules

In [5]:
rules_list = []

if USER_CONFIG_1['enabled']:
    t = USER_CONFIG_1['translation']
    r = USER_CONFIG_1['rotation']
    rules_list.append({'start': 50, 'end': 59, 'aliases': ['obj1', 'gripper'], 'mode': 'T', 'param': t})
    rules_list.append({'start': 50, 'end': 59, 'aliases': ['obj1', 'gripper'], 'mode': 'R', 'param': [r]})

if USER_CONFIG_2['enabled']:
    t = USER_CONFIG_2['translation']
    r = USER_CONFIG_2['rotation']
    rules_list.append({'start': 14, 'end': 20, 'aliases': ['obj2', 'gripper'], 'mode': 'T', 'param': t})
    rules_list.append({'start': 14, 'end': 20, 'aliases': ['obj2', 'gripper'], 'mode': 'R', 'param': [r]})

if USER_CONFIG_3['enabled']:
    t = USER_CONFIG_3['translation']
    r = USER_CONFIG_3['rotation']
    rules_list.append({'start': 102, 'end': 111, 'aliases': ['obj3', 'gripper'], 'mode': 'T', 'param': t})
    rules_list.append({'start': 102, 'end': 111, 'aliases': ['obj3', 'gripper'], 'mode': 'R', 'param': [r]})

if USER_CONFIG_4['enabled']:
    t = USER_CONFIG_4['translation']
    r = USER_CONFIG_4['rotation']
    for start, end, obj in [(25, 32, 'obj2'), (66, 75, 'obj1'), (123, 131, 'obj3')]:
        rules_list.append({'start': start, 'end': end, 'aliases': [obj, 'gripper'], 'mode': 'T', 'param': t})
        rules_list.append({'start': start, 'end': end, 'aliases': [obj, 'gripper'], 'mode': 'R', 'param': [r]})

print(f"✓ Generated {len(rules_list)} rules（in memory）\n")
print("Rules:")
for rule in rules_list:
    mode = rule['mode']
    if mode == 'T':
        param_str = f"T {rule['param'][0]} {rule['param'][1]} {rule['param'][2]}"
    else:
        param_str = f"R {rule['param'][0]}"
    print(f"  {rule['start']}-{rule['end']} , {' '.join(rule['aliases'])} , {param_str}")


✓ Generated 12 rules（in memory）

Rules:
  50-59 , obj1 gripper , T 0.1 0.05 0
  50-59 , obj1 gripper , R 50
  14-20 , obj2 gripper , T -0.02 0 0
  14-20 , obj2 gripper , R 140
  102-111 , obj3 gripper , T -0.02 0 0
  102-111 , obj3 gripper , R -10.0
  25-32 , obj2 gripper , T 0.1 0.05 0
  25-32 , obj2 gripper , R -60
  66-75 , obj1 gripper , T 0.1 0.05 0
  66-75 , obj1 gripper , R -60
  123-131 , obj3 gripper , T 0.1 0.05 0
  123-131 , obj3 gripper , R -60


## 4. Transform Point Clouds

In [6]:
print("=" * 70)
print(" ...")
print("=" * 70)
print("\nin memory\n")

task_dir = "demo_data/bridge"
gripper_cloud_dir = "debug/gripper_clouds"
pose_dir = "demo_data/bridge/pose"
background_dir = "demo_data/bridge/background"
cam_k_path = "demo_data/bridge/cam_K.txt"

task_dir_abs = os.path.abspath(task_dir)
task_name = os.path.basename(task_dir_abs)
mesh_glob_pattern = os.path.join(task_dir_abs, 'mesh', 'obj_*', 'textured.obj')
mesh_paths_sorted = sorted(glob.glob(mesh_glob_pattern))

mesh_entries = []
object_entries = []
for idx, mpath in enumerate(mesh_paths_sorted, start=1):
    alias = f'obj{idx}'
    mesh_entries.append(f"{alias}={mpath}")
    pose_dir_obj = os.path.join(os.getcwd(), 'debug', f"{task_name}_{alias}", 'ob_in_cam')
    object_entries.append(f"{alias}={pose_dir_obj}")

mesh_map_str = ','.join(mesh_entries)
object_map_str = ','.join(object_entries)

pose_dirs = parse_kv_map(object_map_str)
mesh_paths = parse_kv_map(mesh_map_str)
aliases = list(pose_dirs.keys()) + ['gripper']

sample_dir = Path(next(iter(pose_dirs.values())))
frame_ids = sorted(int(p.stem) for p in sample_dir.glob('*.txt'))
frame_min, frame_max = frame_ids[0], frame_ids[-1]
cache = build_transform_cache(rules_list, aliases, frame_min, frame_max, 42)

print(f"Processing frame range: {frame_min} - {frame_max} (total {len(frame_ids)} frames)\n")

color_img = cv2.imread(str(Path(background_dir) / '000000.jpg'))
depth_img = cv2.imread(str(Path(background_dir) / '000000.png'), cv2.IMREAD_UNCHANGED).astype(np.float32) / 1000.0
K = np.loadtxt(cam_k_path).reshape(3, 3)
H, W = color_img.shape[:2]

def depth2xyz(depth):
    i, j = np.indices((H, W))
    fx, fy = K[0,0], K[1,1]
    cx, cy = K[0,2], K[1,2]
    z = depth
    x = (j - cx) * z / fx
    y = (i - cy) * z / fy
    return np.stack((x, y, z), -1)

xyz_map = depth2xyz(depth_img)
valid = depth_img >= 0.001
bg_pts = xyz_map[valid]
bg_clr = color_img[valid][..., ::-1] / 255.0

fixed_deg_z = -3.0
fixed_deg_x = -3.0
fixed_translation_z = 0.02
bg_center = np.mean(bg_pts, axis=0)
R_z = axis_angle_to_mat_z(fixed_deg_z)
centered_points = bg_pts - bg_center
rotated_points = (R_z @ centered_points.T).T
R_x = axis_angle_to_mat_x(fixed_deg_x)
rotated_points = (R_x @ rotated_points.T).T
bg_pts = rotated_points + bg_center
bg_pts[:, 2] += fixed_translation_z

import random
random.seed(42)
bg_deg = random.uniform(-20.0, 20.0)
bg_center = np.mean(bg_pts, axis=0)
R_bg = axis_angle_to_mat_z(bg_deg)
centered = bg_pts - bg_center
bg_pts = (R_bg @ centered.T).T + bg_center

mesh_objs = {al: trimesh.load(mesh_paths[al]) for al in mesh_paths}

pre_sampled = {}
for al in mesh_objs:
    mesh = mesh_objs[al]
    pts_obj, face_inds = trimesh.sample.sample_surface(mesh, 80000)
    colors = np.zeros((len(pts_obj), 3))
    img_np = np.array(mesh.visual.material.image)[:, :, :3]
    tex_h, tex_w = img_np.shape[:2]
    for j in range(len(pts_obj)):
        face_ind = face_inds[j]
        triangle = mesh.vertices[mesh.faces[face_ind]]
        pt = pts_obj[j]
        bary = trimesh.triangles.points_to_barycentric([triangle], [pt])[0]
        uvs = mesh.visual.uv[mesh.faces[face_ind]]
        uv_pt = np.dot(bary, uvs)
        x = uv_pt[0] * (tex_w - 1)
        y = (1 - uv_pt[1]) * (tex_h - 1)
        x0, y0 = int(np.floor(x)), int(np.floor(y))
        x1, y1 = min(x0 + 1, tex_w - 1), min(y0 + 1, tex_h - 1)
        dx, dy = x - x0, y - y0
        ca, cb = img_np[y0, x0], img_np[y1, x0]
        cc, cd = img_np[y0, x1], img_np[y1, x1]
        c = (1 - dx) * (1 - dy) * ca + (1 - dx) * dy * cb + dx * (1 - dy) * cc + dx * dy * cd
        colors[j] = c / 255.0
    pre_sampled[al] = (pts_obj, colors)

glctx = dr.RasterizeCudaContext()
T_CAM_ROBOT = build_camera_to_robot(np.asarray([0.33, 0, 1.64]), -60)
T_ROBOT_CAM = np.linalg.inv(T_CAM_ROBOT)

fixed_rotation_center = calculate_all_objects_center_at_frame(mesh_objs, pose_dirs, frame_max, T_CAM_ROBOT)
rotation_patterns = analyze_rotation_pattern(rules_list, aliases)
first_rotation_completed = {alias: False for alias in aliases}

pointcloud_frames = []
gripper_poses = []

print("frames:")
for idx, fid in enumerate(frame_ids):
    frame_str = f"{fid:06d}"
    merged_pts = np.copy(bg_pts)
    merged_clr = np.copy(bg_clr)

    npy_path = Path(pose_dir) / f"{frame_str}.npy"
    if npy_path.exists():
        T_EE_robot = load_eef_pose_robot_from_npy(npy_path)

        # First compute gripper_T_scene (must be defined before use)
        gripper_t_vec, gripper_deg = cache.get('gripper', {}).get(fid, (np.zeros(3), 0.0))
        T_gripper_translate = compose(np.eye(3), gripper_t_vec)

        if abs(gripper_deg) > 1e-6:
            companion_object, _, interp_alpha = get_gripper_companion_objects(rules_list, fid, frame_min)
            if companion_object:
                obj_center = get_object_center_at_frame(companion_object, mesh_objs, pose_dirs, fid, T_CAM_ROBOT)
                obj_t_vec, _ = cache.get(companion_object, {}).get(fid, (np.zeros(3), 0.0))
                obj_center_after_translation = obj_center + obj_t_vec
                gripper_rotation_center, _ = get_rotation_center_with_interpolation(
                    companion_object, fid, rotation_patterns, first_rotation_completed, frame_min,
                    obj_center_after_translation, fixed_rotation_center)
            else:
                gripper_rotation_center = fixed_rotation_center

            T_gripper_rotate = rotate_around_point(gripper_deg, gripper_rotation_center)
            gripper_T_scene = T_gripper_rotate @ T_gripper_translate
        else:
            gripper_T_scene = T_gripper_translate

        # Now use gripper_T_scene (after it's defined)
        T_EE_robot_transformed = gripper_T_scene @ T_EE_robot

        gripper_poses.append({
            'frame': fid,
            'T_EE_robot': T_EE_robot_transformed.copy(),
            'position': T_EE_robot_transformed[:3, 3].copy()
        })

        pcd_path = Path(gripper_cloud_dir) / f"{frame_str}.ply"
        if pcd_path.exists():
            pcd = o3d.io.read_point_cloud(str(pcd_path))
            pcd.transform(T_CAM_ROBOT)
            pcd.transform(gripper_T_scene)
            pcd.transform(T_ROBOT_CAM)
            merged_pts = np.vstack([merged_pts, np.asarray(pcd.points)])
            if pcd.has_colors():
                merged_clr = np.vstack([merged_clr, np.asarray(pcd.colors)])
            else:
                merged_clr = np.vstack([merged_clr, np.tile([0.9, 0.9, 0.0], (len(pcd.points), 1))])

    for al in aliases:
        if al == 'gripper':
            continue
        if al not in pose_dirs:
            continue

        pose_path = Path(pose_dirs[al]) / f"{frame_str}.txt"
        if not pose_path.exists():
            continue

        t_vec, deg = cache.get(al, {}).get(fid, (np.zeros(3), 0.0))
        T_translate = compose(np.eye(3), t_vec)

        if abs(deg) > 1e-6:
            object_original_center = get_object_center_at_frame(al, mesh_objs, pose_dirs, fid, T_CAM_ROBOT)
            object_center_after_translation = object_original_center + t_vec
            rotation_center, center_status = get_rotation_center_with_interpolation(
                al, fid, rotation_patterns, first_rotation_completed, frame_min,
                object_center_after_translation, fixed_rotation_center)
            T_rotate = rotate_around_point(deg, rotation_center)
            T_scene = T_rotate @ T_translate

            if al in rotation_patterns and len(rotation_patterns[al]) > 1:
                second_rotation_start = rotation_patterns[al][1][0]
                if fid >= second_rotation_start and not first_rotation_completed.get(al, False):
                    first_rotation_completed[al] = True
            elif al in rotation_patterns and len(rotation_patterns[al]) == 1:
                first_rotation_end = rotation_patterns[al][0][1]
                if fid > first_rotation_end and not first_rotation_completed.get(al, False):
                    first_rotation_completed[al] = True
        else:
            T_scene = T_translate

        T_obj_cam = np.loadtxt(pose_path)
        T_obj_robot = T_CAM_ROBOT @ T_obj_cam
        T_obj_robot_new = T_scene @ T_obj_robot
        T_obj_cam_new = T_ROBOT_CAM @ T_obj_robot_new

        pts_obj, clr_base = pre_sampled[al]
        hpts = np.hstack([pts_obj, np.ones((len(pts_obj), 1))])
        pts = (T_obj_cam_new @ hpts.T)[:3, :].T

        proj_h = (K @ pts.T).T
        proj = proj_h[:, :2] / (proj_h[:, 2:] + 1e-9)
        valid = (proj[:, 0] >= 0) & (proj[:, 0] < W) & (proj[:, 1] >= 0) & (proj[:, 1] < H) & (proj_h[:, 2] > 0)
        clr = np.copy(clr_base)
        clr[~valid] = [0.5, 0.5, 0.5]

        merged_pts = np.vstack([merged_pts, pts])
        merged_clr = np.vstack([merged_clr, clr])

    pcd_frame = o3d.geometry.PointCloud()
    pcd_frame.points = o3d.utility.Vector3dVector(merged_pts)
    pcd_frame.colors = o3d.utility.Vector3dVector(merged_clr)
    pointcloud_frames.append(pcd_frame)

    progress = (idx + 1) / len(frame_ids) * 100
    bar_length = 40
    filled = int(bar_length * (idx + 1) / len(frame_ids))
    bar = "#" * filled + "-" * (bar_length - filled)
    import sys
    sys.stdout.write(f"\r  [{bar}] {progress:5.1f}% ({idx + 1}/{len(frame_ids)})")
    sys.stdout.flush()

print(f"\n\n✅ Point cloud generation complete！")
print(f"Generated {len(pointcloud_frames)} point cloud objects（in memory）")
print(f"Memory estimate: ~{len(pointcloud_frames) * len(merged_pts) * 24 / (1024**2):.1f} MB")


 ...

in memory

Processing frame range: 0 - 138 (total 139 frames)



W1004 00:05:43.253000 2087 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1004 00:05:43.253000 2087 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


frames:
  [########################################] 100.0% (139/139)

✅ Point cloud generation complete！
Generated 139 point cloud objects（in memory）
Memory estimate: ~3019.6 MB


## 5. Apply Z-Buffer Filtering

In [7]:
try:
    import cupy as cp
    CUDA_AVAILABLE = True
except ImportError:
    CUDA_AVAILABLE = False
    print("Warning: CuPy not available, Z-buffer filtering will be skipped")

import time
import sys

CUDA_KERNEL_SOURCE = r"""
extern "C" __global__
void splatting_kernel(
    const float* __restrict__ valid_pts,
    float* depth_buffer,
    int* index_buffer,
    const int num_points,
    const float fx,
    const float fy,
    const float cx,
    const float cy,
    const int width,
    const int height,
    const int splat_size
)
{
    const int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= num_points) {
        return;
    }

    const int pt_offset = tid * 3;
    const float x = valid_pts[pt_offset + 0];
    const float y = valid_pts[pt_offset + 1];
    const float z = valid_pts[pt_offset + 2];

    const float u = (fx * x / z) + cx;
    const float v = (fy * y / z) + cy;

    const int u_center = roundf(u);
    const int v_center = roundf(v);

    for (int r_offset = -splat_size; r_offset <= splat_size; ++r_offset) {
        for (int c_offset = -splat_size; c_offset <= splat_size; ++c_offset) {

            const int cur_v = v_center + r_offset;
            const int cur_u = u_center + c_offset;

            if (cur_v >= 0 && cur_v < height && cur_u >= 0 && cur_u < width) {

                const int pixel_idx = cur_v * width + cur_u;
                float current_depth_in_buffer = depth_buffer[pixel_idx];

                while (z < current_depth_in_buffer) {
                    unsigned int current_depth_as_int = __float_as_int(current_depth_in_buffer);

                    unsigned int previous_val_as_int = atomicCAS(
                        (unsigned int*)&depth_buffer[pixel_idx],
                        current_depth_as_int,
                        __float_as_int(z)
                    );

                    if (previous_val_as_int == current_depth_as_int) {
                        index_buffer[pixel_idx] = tid;
                        break;
                    }

                    current_depth_in_buffer = __int_as_float(previous_val_as_int);
                }
            }
        }
    }
}
"""

_splatting_kernel = None

def _zbuffer_filter_pcd_cuda(pcd, K, image_size, splat_size=1):
    global _splatting_kernel

    if _splatting_kernel is None:
        if not cp:
            raise RuntimeError("CuPy is not available in this worker process.")
        try:
            _splatting_kernel = cp.RawKernel(CUDA_KERNEL_SOURCE, 'splatting_kernel')
        except Exception as e:
            raise RuntimeError(f"Failed to compile CUDA kernel in worker: {e}")

    width, height = image_size
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    if not pcd.has_points():
        return pcd

    pts_gpu = cp.asarray(pcd.points, dtype=cp.float32)
    if pcd.has_colors():
        cols_gpu = cp.asarray(pcd.colors, dtype=cp.float32)
    else:
        cols_gpu = cp.zeros_like(pts_gpu)

    z = pts_gpu[:, 2]
    u = (fx * pts_gpu[:, 0] / z) + cx
    v = (fy * pts_gpu[:, 1] / z) + cy
    valid_mask = (u >= -splat_size) & (u < width + splat_size) & (v >= -splat_size) & (v < height + splat_size) & (z > 1e-6)

    pts_valid = pts_gpu[valid_mask]
    cols_valid = cols_gpu[valid_mask]
    num_valid_points = pts_valid.shape[0]

    if num_valid_points == 0:
        return o3d.geometry.PointCloud()

    depth_buffer = cp.full((height * width,), cp.inf, dtype=cp.float32)
    index_buffer = cp.full((height * width,), -1, dtype=cp.int32)

    block_size = 256
    grid_size = (num_valid_points + block_size - 1) // block_size

    kernel_args = (
        pts_valid, depth_buffer, index_buffer, num_valid_points,
        fx, fy, cx, cy, width, height, splat_size
    )

    _splatting_kernel((grid_size,), (block_size,), kernel_args)

    visible_original_indices = index_buffer[index_buffer != -1]
    unique_indices = cp.unique(visible_original_indices)

    if unique_indices.size == 0:
        return o3d.geometry.PointCloud()

    pts_out_gpu = pts_valid[unique_indices]
    cols_out_gpu = cols_valid[unique_indices]

    pcd_out = o3d.geometry.PointCloud()
    pcd_out.points = o3d.utility.Vector3dVector(cp.asnumpy(pts_out_gpu))
    pcd_out.colors = o3d.utility.Vector3dVector(cp.asnumpy(cols_out_gpu))

    return pcd_out



print("Z-buffer filtering: Loading camera intrinsics")
K = np.loadtxt(cam_k_path).reshape(3, 3).astype(np.float32)
image_size = (1280, 720)
splat_size = 1

if CUDA_AVAILABLE:
    print(f"Applying Z-buffer filtering to {len(pointcloud_frames)} frames")
    print(f"Image size: {image_size[0]}x{image_size[1]}")
    print("Processing using single thread...")

    total_start_time = time.time()

    # Process in-place to save memory
    for idx in range(len(pointcloud_frames)):
        pcd_filtered = _zbuffer_filter_pcd_cuda(pointcloud_frames[idx], K, image_size, splat_size)
        pointcloud_frames[idx] = pcd_filtered  # Replace original with filtered

        progress = (idx + 1) / len(pointcloud_frames) * 100
        bar_length = 40
        filled = int(bar_length * (idx + 1) / len(pointcloud_frames))
        bar = "#" * filled + "-" * (bar_length - filled)
        sys.stdout.write(f"\r  [{bar}] {progress:5.1f}% ({idx + 1}/{len(pointcloud_frames)})")
        sys.stdout.flush()

    print()
    total_time = time.time() - total_start_time
    print(f"Total time: {total_time:.2f} seconds")

    avg_points = np.mean([len(pcd.points) for pcd in pointcloud_frames])
    print(f"Z-buffer filtering complete")
    print(f"Average points per frame: {avg_points:.0f}")
else:
    print("Skipping Z-buffer filtering (CuPy not available)")


Z-buffer filtering: Loading camera intrinsics
Applying Z-buffer filtering to 139 frames
Image size: 1280x720
Processing using single thread...
  [########################################] 100.0% (139/139)
Total time: 99.64 seconds
Z-buffer filtering complete
Average points per frame: 423225


## 6. Render Point Cloud Video

In [8]:
import os
import sys
import numpy as np

if os.environ.get("DISPLAY", "") == "":
    os.environ.setdefault("EGL_PLATFORM", "surfaceless")
os.environ.pop("OPEN3D_CPU_RENDERING", None)

import open3d as o3d

try:
    import imageio.v2 as imageio
except ImportError:
    import imageio

def render_pointclouds_to_video(pcd_list, output_file, fps=30, image_size=(1920, 1080), translate=(0.0, 0.0, 0.3)):
    """
    in memoryRender Video
    """
    if not pcd_list:
        print(": ")
        return False

    print(f"\nRendering {len(pcd_list)} frames")
    print(f"Output: {output_file}")
    print(f"Resolution: {image_size[0]}x{image_size[1]}, FPS: {fps}")

    width, height = image_size

    renderer = o3d.visualization.rendering.OffscreenRenderer(width, height)
    scene = renderer.scene
    scene.set_background([0, 0, 0, 1])

    material = o3d.visualization.rendering.MaterialRecord()
    material.shader = "defaultUnlit"
    material.point_size = 3.0

    forward = np.array([0.0, 0.0, 1.0], dtype=float)
    up = np.array([0.0, -1.0, 0.0], dtype=float)
    translate_vec = np.asarray(translate, dtype=float)
    eye = translate_vec
    target = translate_vec + forward

    scene.add_geometry("pcd", pcd_list[0], material)
    cam = scene.camera
    cam.look_at(target.tolist(), eye.tolist(), up.tolist())

    writer = imageio.get_writer(
        str(output_file),
        fps=fps,
        format="ffmpeg",
        codec="libx264",
        ffmpeg_params=["-preset", "medium"],
    )

    total_frames = len(pcd_list)
    print(f"\nRendering...")

    try:
        for idx, pcd in enumerate(pcd_list):
            if idx > 0:
                scene.remove_geometry("pcd")
                scene.add_geometry("pcd", pcd, material)

            cam.look_at(target.tolist(), eye.tolist(), up.tolist())
            img_o3d = renderer.render_to_image()
            img_np = np.asarray(img_o3d)
            writer.append_data(img_np[:, :, :3])

            progress = (idx + 1) / total_frames * 100
            bar_length = 40
            filled = int(bar_length * (idx + 1) / total_frames)
            bar = "#" * filled + "-" * (bar_length - filled)
            sys.stdout.write(f"\r  Progress: [{bar}] {progress:5.1f}% ({idx + 1}/{total_frames})")
            sys.stdout.flush()

        print()

    finally:
        writer.close()

    print(f"\n✅ Rendering: {output_file}")
    return True

output_video = "disturbed_cloud_demo.mp4"
success = render_pointclouds_to_video(
    pointcloud_frames,
    output_video,
    fps=30,
    image_size=(1920, 1080),
    translate=(0.0, 0.0, 0.3)
)

if success:
    video_size = os.path.getsize(output_video) / (1024 * 1024)
    print(f"Video file size: {video_size:.2f} MB")
else:
    print("❌ Rendering")



Rendering 139 frames
Output: disturbed_cloud_demo.mp4
Resolution: 1920x1080, FPS: 30
[Open3D INFO] EGL headless mode enabled.



Rendering...
  Progress: [########################################] 100.0% (139/139)

✅ Rendering: disturbed_cloud_demo.mp4
Video file size: 6.59 MB


## 7. Render Gripper Trajectory Video

In [9]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib
matplotlib.use('Agg')

print(f"Rendering gripper trajectory video from {len(gripper_poses)} poses")

trajectory_frames = []

positions = np.array([p['position'] for p in gripper_poses])
x_min, x_max = positions[:, 0].min() - 0.1, positions[:, 0].max() + 0.1
y_min, y_max = positions[:, 1].min() - 0.1, positions[:, 1].max() + 0.1
z_min, z_max = positions[:, 2].min() - 0.1, positions[:, 2].max() + 0.1

arrow_scale = 0.03
skip_frames = max(1, len(gripper_poses) // 50)

for idx, pose_data in enumerate(gripper_poses):
    fig = plt.figure(figsize=(8, 8), dpi=120)
    ax = fig.add_subplot(111, projection='3d')

    history_positions = positions[:idx+1]

    ax.plot(history_positions[:, 0],
            history_positions[:, 1],
            history_positions[:, 2],
            'b-', linewidth=2.5, alpha=0.7, label='Trajectory Path')

    ax.scatter(history_positions[:, 0],
               history_positions[:, 1],
               history_positions[:, 2],
               c=np.arange(idx+1), cmap='viridis', s=20, alpha=0.6)

    if idx == 0:
        ax.scatter([positions[0, 0]], [positions[0, 1]], [positions[0, 2]],
                   c='g', s=150, marker='^', label='Start', edgecolors='black', linewidths=2)

    for i in range(0, idx+1, skip_frames):
        T = gripper_poses[i]['T_EE_robot']
        origin = T[:3, 3]

        z_axis = T[:3, 2]

        ax.quiver(origin[0], origin[1], origin[2],
                  z_axis[0]*arrow_scale, z_axis[1]*arrow_scale, z_axis[2]*arrow_scale,
                  color='red', arrow_length_ratio=0.3, linewidth=1.5, alpha=0.6)

    T_current = pose_data['T_EE_robot']
    origin_current = T_current[:3, 3]

    x_axis = T_current[:3, 0] * arrow_scale * 1.5
    y_axis = T_current[:3, 1] * arrow_scale * 1.5
    z_axis = T_current[:3, 2] * arrow_scale * 1.5

    ax.quiver(origin_current[0], origin_current[1], origin_current[2],
              x_axis[0], x_axis[1], x_axis[2],
              color='red', arrow_length_ratio=0.3, linewidth=3, label='X-axis')
    ax.quiver(origin_current[0], origin_current[1], origin_current[2],
              y_axis[0], y_axis[1], y_axis[2],
              color='green', arrow_length_ratio=0.3, linewidth=3, label='Y-axis')
    ax.quiver(origin_current[0], origin_current[1], origin_current[2],
              z_axis[0], z_axis[1], z_axis[2],
              color='blue', arrow_length_ratio=0.3, linewidth=3, label='Z-axis (Forward)')

    ax.scatter([origin_current[0]], [origin_current[1]], [origin_current[2]],
               c='gold', s=200, marker='o', edgecolors='black', linewidths=2,
               label='Current Position', zorder=10)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_zlim(z_min, z_max)
    ax.set_xlabel('X (m)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Y (m)', fontsize=10, fontweight='bold')
    ax.set_zlabel('Z (m)', fontsize=10, fontweight='bold')

    position_str = f"({origin_current[0]:.3f}, {origin_current[1]:.3f}, {origin_current[2]:.3f})"
    ax.set_title(f'Gripper End-Effector Trajectory\nFrame {pose_data["frame"]} | Position: {position_str}',
                 fontsize=11, fontweight='bold')

    ax.legend(loc='upper left', fontsize=8)
    ax.view_init(elev=25, azim=45 + idx * 0.5)
    ax.grid(True, alpha=0.3)

    ax.set_box_aspect([1, 1, 1])

    fig.canvas.draw()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
    trajectory_frames.append(img)

    plt.close(fig)

    progress = (idx + 1) / len(gripper_poses) * 100
    bar_length = 40
    filled = int(bar_length * (idx + 1) / len(gripper_poses))
    bar = "#" * filled + "-" * (bar_length - filled)
    sys.stdout.write(f"\r  [{bar}] {progress:5.1f}% ({idx + 1}/{len(gripper_poses)})")
    sys.stdout.flush()

print(f"\n\nTrajectory frames rendered: {len(trajectory_frames)}")

try:
    import imageio.v2 as imageio
except ImportError:
    import imageio

trajectory_video = "gripper_trajectory.mp4"
writer = imageio.get_writer(
    trajectory_video,
    fps=30,
    format="ffmpeg",
    codec="libx264",
    ffmpeg_params=["-preset", "medium"],
)

for frame in trajectory_frames:
    writer.append_data(frame)

writer.close()

traj_size = os.path.getsize(trajectory_video) / (1024 * 1024)
print(f"Trajectory video saved: {trajectory_video} ({traj_size:.2f} MB)")


Rendering gripper trajectory video from 139 poses
  [----------------------------------------]   1.4% (2/139)

/tmp/ipython-input-1954621270.py:87: MatplotlibDeprecationWarning: The tostring_rgb function was deprecated in Matplotlib 3.8 and will be removed in 3.10. Use buffer_rgba instead.
  img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)


  [########################################] 100.0% (139/139)

Trajectory frames rendered: 139
Trajectory video saved: gripper_trajectory.mp4 (0.63 MB)


## 8. Concatenate Videos Side by Side

In [10]:
print("Concatenating point cloud video and trajectory video side by side...")

try:
    import imageio.v2 as imageio
except ImportError:
    import imageio

pcd_video = output_video
traj_video = "gripper_trajectory.mp4"
output_combined = "combined_demo.mp4"

pcd_reader = imageio.get_reader(pcd_video)
traj_reader = imageio.get_reader(traj_video)

pcd_meta = pcd_reader.get_meta_data()
traj_meta = traj_reader.get_meta_data()

print(f"Point cloud video: {pcd_meta['size']}, {pcd_meta['fps']} fps")
print(f"Trajectory video: {traj_meta['size']}, {traj_meta['fps']} fps")

writer = imageio.get_writer(
    output_combined,
    fps=30,
    format="ffmpeg",
    codec="libx264",
    ffmpeg_params=["-preset", "medium"],
)

for idx, (pcd_frame, traj_frame) in enumerate(zip(pcd_reader, traj_reader)):
    pcd_h, pcd_w = pcd_frame.shape[:2]
    traj_h, traj_w = traj_frame.shape[:2]

    target_h = max(pcd_h, traj_h)

    if pcd_h < target_h:
        pad = target_h - pcd_h
        pcd_frame = np.pad(pcd_frame, ((pad//2, pad-pad//2), (0, 0), (0, 0)), mode='constant')
    elif traj_h < target_h:
        pad = target_h - traj_h
        traj_frame = np.pad(traj_frame, ((pad//2, pad-pad//2), (0, 0), (0, 0)), mode='constant')

    combined_frame = np.hstack([pcd_frame, traj_frame])
    writer.append_data(combined_frame)

    progress = (idx + 1) / len(trajectory_frames) * 100
    bar_length = 40
    filled = int(bar_length * (idx + 1) / len(trajectory_frames))
    bar = "#" * filled + "-" * (bar_length - filled)
    sys.stdout.write(f"\r  [{bar}] {progress:5.1f}% ({idx + 1}/{len(trajectory_frames)})")
    sys.stdout.flush()

writer.close()
pcd_reader.close()
traj_reader.close()

print()

combined_size = os.path.getsize(output_combined) / (1024 * 1024)
print(f"\nCombined video saved: {output_combined} ({combined_size:.2f} MB)")

output_video = output_combined


Concatenating point cloud video and trajectory video side by side...
Point cloud video: (1920, 1088), 30.0 fps
Trajectory video: (960, 960), 30.0 fps
  [########################################] 100.0% (139/139)

Combined video saved: combined_demo.mp4 (6.96 MB)


## 9. Play Combined Video

In [11]:
from IPython.display import Video
Video("combined_demo.mp4", width=1200, embed=True)

Output hidden; open in https://colab.research.google.com to view.

## 10. Cleanup

In [12]:
import gc

if 'pointcloud_frames' in dir():
    print(f"Releasing {len(pointcloud_frames)} point cloud objects...")
    del pointcloud_frames
    gc.collect()
    print("✓ Releasing")

for var in ['mesh_objs', 'pre_sampled', 'bg_pts', 'bg_clr']:
    if var in dir():
        exec(f"del {var}")

gc.collect()
print("✓ Cleanup")


Releasing 139 point cloud objects...
✓ Releasing
✓ Cleanup
